# V2 attribute-schema catalog profile

This notebook reproduces the catalog coverage observations used in `docs/v2_attribute_schema.md`. Regex matches are rough lower-bound indicators of explicit evidence, not ground-truth attribute labels.

In [ ]:
import json
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "catalog.jsonl").is_file():
            return candidate
    raise FileNotFoundError("Could not locate data/catalog.jsonl")


project_root = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(project_root))

from retrieval.catalog import category_group

catalog_path = project_root / "data" / "catalog.jsonl"
print(f"Catalog: {catalog_path}")

## Group composition and structured-detail coverage

In [ ]:
group_counts = Counter()
leaf_counts = defaultdict(Counter)
detail_key_counts = defaultdict(Counter)

with catalog_path.open(encoding="utf-8") as handle:
    for line in handle:
        product = json.loads(line)
        categories = [str(value) for value in product.get("categories") or []]
        group = category_group(categories)
        group_counts[group] += 1
        if categories:
            leaf_counts[group][categories[-1]] += 1
        details = product.get("details")
        if isinstance(details, dict):
            for key, value in details.items():
                if value not in (None, "", [], {}):
                    detail_key_counts[group][str(key)] += 1

for group in ("clothing", "shoes", "jewelry"):
    count = group_counts[group]
    print(f"\n{group}: {count:,} products")
    print("top leaves:", leaf_counts[group].most_common(15))
    print("top details:")
    for key, present in detail_key_counts[group].most_common(25):
        print(f"  {key:35} {present:6,} ({present / count:6.1%})")

## Explicit factual-evidence indicators

The patterns intentionally look for mechanisms and specifications rather than generic claims such as *comfortable* or *high quality*.

In [ ]:
evidence_patterns = {
    "cushioning": r"\b(?:cushion(?:ed|ing)?|memory foam|foam footbed|shock absor(?:b|bing|ption))\b",
    "breathability": r"\b(?:breathable|ventilat(?:ed|ion)|mesh upper|moisture[- ]wicking)\b",
    "stretch_flexibility": r"\b(?:stretch(?:y)?|flexib(?:le|ility)|elastic(?:ated)?)\b",
    "support": r"\b(?:arch support|ankle support|supportive|orthotic|stability)\b",
    "reinforced_construction": r"\b(?:reinforced|double[- ]stitched|triple[- ]stitched|ripstop|abrasion[- ]resistant|heavy[- ]duty)\b",
    "water_protection": r"\b(?:waterproof|water[- ]resistant|water resistant|water resistance)\b",
    "weather_insulation": r"\b(?:insulated|thermal|fleece[- ]lined|windproof|snowproof)\b",
    "sun_protection": r"\b(?:uv400|uv protection|sun protection|upf\s*\d+)\b",
    "closure": r"\b(?:zipper|zip closure|button closure|lace[- ]up|hook[- ]and[- ]loop|buckle|drawstring|snap closure|slip[- ]on)\b",
    "pockets": r"\b(?:pocket|pockets)\b",
    "personalization": r"\b(?:personalized|custom(?:ized)?|engraved|engraving|monogram(?:med)?|name necklace)\b",
    "gemstone": r"\b(?:diamond|cubic zirconia|zirconia|ruby|sapphire|emerald|opal|pearl|amethyst|turquoise|moissanite|birthstone)\b",
    "gemstone_spec": r"\b(?:carat|clarity|lab[- ](?:grown|created)|natural (?:diamond|gemstone|stone)|treated|untreated|conflict[- ]free)\b",
    "watch_movement": r"\b(?:quartz movement|automatic movement|mechanical movement|self[- ]winding|chronograph)\b",
    "watch_resistance": r"\b(?:water resistant to \d+|water resistance depth|\d+\s*(?:atm|meters?) water)\b",
    "metal": r"\b(?:sterling silver|stainless steel|yellow gold|white gold|rose gold|platinum|titanium|brass|alloy|gold[- ]plated|silver[- ]plated)\b",
    "care": r"\b(?:machine wash(?:able)?|hand wash|dry clean|tumble dry)\b",
}
compiled_patterns = {
    name: re.compile(pattern, re.IGNORECASE)
    for name, pattern in evidence_patterns.items()
}
evidence_counts = defaultdict(Counter)

with catalog_path.open(encoding="utf-8") as handle:
    for line in handle:
        product = json.loads(line)
        categories = [str(value) for value in product.get("categories") or []]
        group = category_group(categories)
        details = product.get("details")
        details = details if isinstance(details, dict) else {}
        sources = [product.get("title") or "", *(product.get("features") or [])]
        description = product.get("description")
        if isinstance(description, list):
            sources.extend(description)
        elif description:
            sources.append(description)
        sources.extend(f"{key}: {value}" for key, value in details.items())
        text = " ".join(str(value) for value in sources)
        for name, pattern in compiled_patterns.items():
            if pattern.search(text):
                evidence_counts[group][name] += 1

for group in ("clothing", "shoes", "jewelry"):
    count = group_counts[group]
    print(f"\n{group}: {count:,} products")
    for name, present in evidence_counts[group].most_common():
        print(f"  {name:25} {present:6,} ({present / count:6.1%})")